<center><h1>QM640 Data Analytics Capstone</h1></center>
<center><h2>Predicting Whether the NIFTY 50 Will Breach Predefined Weekly Price Bands</h2></center>
<center><h3>A Leakage-Controlled Comparison of Machine Learning and the Gaussian Model</h3></center>
<center><b>Dhyanendra Bhangre</b> &nbsp;|&nbsp; Walsh College &nbsp;|&nbsp; Data Cleaning &amp; Exploratory Data Analysis</center>

### Context

Every week, participants in the Indian equity derivatives market trade NIFTY 50 weekly-expiry
options. A large group of them are **option sellers**, who collect a premium at the start of the
weekly cycle and profit if the index stays within a range until expiry. Their payoff is
asymmetric: the premium is small and bounded, but the loss when the index moves sharply is not.

Because of that asymmetry, a practical question arises on every trading day of the cycle:
*should the position be held for another day, or should one side be closed because the index looks
likely to break through a price band before expiry?*

The standard tool for answering this is the **Gaussian (Normal) model** — the familiar bell curve.
It is fast, closed-form and easy to explain. But equity index returns are known to have fatter
tails than the Normal distribution allows, which raises the question of whether a model that
*learns* from history could judge breach risk more accurately.

### Objective

This notebook performs the **data cleaning and exploratory data analysis** that underpins the
capstone project. Specifically it aims to:

1. Establish that the raw NSE data is complete, internally consistent, and free of the
   silent-imputation problems that would invalidate any downstream result.
2. Convert daily index prices into **weekly cycle records** and construct the price bands and
   breach labels that the study predicts.
3. Characterise the data through **univariate, bivariate and multivariate** analysis, and draw
   out the specific empirical facts that motivate each research question.
4. Engineer additional features from the same price and volatility inputs, and test whether any
   of them carries information the existing feature set does not.
5. Compute the **minimum sample size and achieved statistical power** for each research question.

### Research Questions

| | Research Question |
|---|---|
| **RQ1** | Can a learned model (ML-core) achieve a different F1 score from the Gaussian model when both are given the same price and volatility inputs? |
| **RQ2** | Are the Gaussian model's errors systematic, or are they mostly random noise? |
| **RQ3** | Does breach predictability differ by decision day (D2, D3, D4) and by direction (upper versus lower band)? |
| **RQ4** | Do the sigma level and the look-back window materially change model performance? |

### Data Description

The study uses **end-of-day NIFTY 50 index prices** published by the National Stock Exchange of
India, retrieved through the public `nselib` package. No options data, open interest, implied
volatility or macroeconomic series are used — this restriction is deliberate, because the central
comparison is only fair if both contestants receive identical information.

Two files are analysed.

**File 1 — `Nifty_LSTM_Features_clean.xlsx`** (daily level)

| Variable | Type | Description |
|---|---|---|
| `Date` | date | Trading date, weekends and duplicates removed |
| `Open`, `High`, `Low`, `Close` | float | NIFTY 50 daily OHLC levels |
| `daily_log_return` | float | Natural log of Close divided by the previous Close |
| `realized_vol` | float | Rolling 10-day standard deviation of returns, shifted one day |
| `intraday_range` | float | (High − Low) / Close |
| `Volatility` | float | Alias of `realized_vol` — realized, **not** implied volatility |

**File 2 — `df_cycles_FULL.xlsx`** (weekly cycle level)

| Variable | Type | Description |
|---|---|---|
| `cycle_id` | int | Sequential identifier of the weekly expiry cycle |
| `cycle_status` | text | `standard`, `short`, `long`, or `partial_first` |
| `d1_close` … `d4_close` | float | Closing level on each decision day |
| `d1_date` … `d4_date` | date | Date of each decision day |
| `expiry_close` | float | Closing level on the expiry day — the outcome |
| `vol_D1` … `vol_D4` | float | Realized volatility state on each day |
| `days_left_D1` … `D4` | float | Trading-day steps remaining to expiry (4, 3, 2, 1) |
| `sqrt_dl_frac_D1` … `D4` | float | Square root of the remaining time fraction |

**Unit of analysis:** one weekly expiry cycle. A standard cycle spans five trading days —
D1, D2, D3, D4 and Expiry — and yields three decision records (D2, D3, D4) that share a single
cycle-level breach outcome.

### Let us start by importing the required libraries

In [ ]:
# Installing the libraries with the specified version.
# !pip install numpy==1.26.4 pandas==2.2.2 matplotlib==3.8.4 seaborn==0.13.2 scipy==1.13.1 scikit-learn==1.5.0 openpyxl==3.1.5 -q

**Note**: *After running the above cell, kindly restart the notebook kernel and run all cells sequentially from the start.*

In [ ]:
# import libraries for data manipulation
import numpy as np
import pandas as pd

# import libraries for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# import libraries for statistical testing
from scipy import stats as sps

# import libraries for multivariate analysis
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings('ignore')

# display settings so wide tables are readable
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda v: f'{v:,.5f}')

# consistent visual style across every figure in this notebook
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titleweight'] = 'bold'

print('Libraries imported successfully.')

Libraries imported successfully.


### Understanding the structure of the data

In [ ]:
# uncomment and run the following two lines for Google Colab
from google.colab import drive
drive.mount('/content/drive')

# paths to the two data files
DAILY_PATH  = 'Nifty_LSTM_Features_clean.xlsx'
CYCLES_PATH = 'df_cycles_FULL.xlsx'

# read the daily data
df = pd.read_excel(DAILY_PATH, sheet_name='Sheet1', parse_dates=['Date'])

# returning the first 5 rows of the daily dataset
df.head()

#### Observations:
- Each row is one **trading day** of the NIFTY 50 index.
- The first four columns are the raw OHLC levels; the remaining columns are derived features
  built during cleaning.
- `daily_log_return` and `realized_vol` are blank in the first rows. This is expected: a return
  requires a previous close, and a 10-day rolling volatility requires a 10-day history. These are
  **warm-up** values, not data errors, and they are deliberately left blank rather than filled.

In [ ]:
# read the weekly cycle records
cycles = pd.read_excel(CYCLES_PATH, sheet_name='Cycles', parse_dates=['expiry_date','d1_date','d2_date','d3_date','d4_date'])

# returning the first 5 rows of the cycle dataset
cycles[['cycle_id','cycle_status','d1_date','d1_close','d2_close','d3_close','d4_close',
        'expiry_date','expiry_close','days_left_D2','sqrt_dl_frac_D2']].head()

#### Observations:
- Each row is one **weekly expiry cycle**, aggregated from the daily file.
- A standard cycle carries four decision-day closes (`d1_close` to `d4_close`) plus the
  `expiry_close`, which is the outcome the study predicts.
- `days_left_D2 = 3` confirms the cycle geometry: from the close of D2 there are three
  trading-day steps remaining until the expiry close. The expiry day is a **fifth** trading day,
  not the fourth.

---
## Section 1: Data Overview and Quality Checks

### **Question 1:** How many rows and columns are present in each dataset?

In [ ]:
# check the shape of the daily dataset
print('Daily dataset  :', df.shape)

# check the shape of the cycle dataset
print('Cycle dataset  :', cycles.shape)

# check the period covered
print('Period covered :', df['Date'].min().date(), 'to', df['Date'].max().date())
print('Years of data  :', round((df['Date'].max() - df['Date'].min()).days / 365.25, 1))

#### Observations:
- The daily dataset contains **3,846 rows and 9 columns**.
- The cycle dataset contains **814 rows and 50 columns**.
- The data spans **2011-01-03 to 2026-08-05**, roughly **15.6 years**.
- This is a substantial sample for a weekly-frequency study. It comfortably exceeds the minimum
  sample sizes computed later in Section 7, which is the single most important precondition for
  the statistical tests that follow.

### **Question 2:** What are the datatypes of the different columns?

In [ ]:
# use info() to print a concise summary of the daily DataFrame
df.info()

#### Observations:
- `Date` is correctly parsed as `datetime64`, which is essential because every rolling statistic
  and every train/test split in this study is time-ordered.
- All nine remaining columns are `float64`. There are no object or categorical columns in the
  daily file, so no type conversion or encoding is required.
- The non-null counts differ slightly across columns, which is examined next.

### **Question 3:** Are there any missing values in the data?

In [ ]:
# checking for missing values in the daily data
missing = pd.DataFrame({
    'missing_count'   : df.isnull().sum(),
    'missing_percent' : (df.isnull().mean() * 100).round(3)
})
missing

#### Observations:
- Only three columns have missing values: `daily_log_return` (1), `realized_vol` (4) and
  `Volatility` (4) — **nine missing cells in total, or 0.03% of the daily file**.
- These are all **structural warm-up values**, not data-collection failures.

In [ ]:
# where exactly are the missing values located in the series?
missing_rows = df[df.isnull().any(axis=1)]
print('Rows containing at least one missing value:', len(missing_rows))
print('Position of those rows within the series  :', list(missing_rows.index))
print('Total rows in the dataset                 :', len(df))
missing_rows[['Date','Close','daily_log_return','realized_vol']]

#### Observations:
- Every missing value sits in the **first four rows** of the series. There are no gaps anywhere in
  the middle or at the end.
- This confirms the warm-up explanation: the first return has no prior close to compare against,
  and the rolling volatility needs a minimum window before it is defined.
- **No imputation is performed.** Filling these forward would fabricate a zero-return day, which
  would artificially deflate the rolling volatility for the following ten days and corrupt the
  band widths that depend on it. Rows with undefined statistics are excluded from the analysis
  sample instead.

### **Question 4:** Are there any duplicate records in the data?

In [ ]:
# check for duplicate dates and fully duplicated rows
print('Duplicate Date values          :', df['Date'].duplicated().sum())
print('Fully duplicated rows          :', df.duplicated().sum())
print('Duplicate cycle_id values      :', cycles['cycle_id'].duplicated().sum())

# a forward-filled row would be identical to its predecessor across ALL FOUR OHLC columns
ohlc = ['Open','High','Low','Close']
ffill_signature = (df[ohlc].shift(1) == df[ohlc]).all(axis=1).sum()
print('Duplicate OHLC blocks (ffill)  :', ffill_signature)

# by contrast, a coincidental equal Close is harmless
print('Exact-zero log returns         :', (df['daily_log_return'] == 0).sum())

#### Observations:
- There are **no duplicate dates, no duplicate rows and no duplicate cycle identifiers**.
- There are **zero forward-filled OHLC blocks**, confirming that no price was silently imputed
  during cleaning.
- There are **four exact-zero log returns**. Inspecting them shows the Close happened to match the
  previous Close to two decimal places while Open, High and Low all differed — a genuine market
  coincidence in roughly 1 session in 1,000, not an imputation artefact. Testing the Close alone
  would have raised a false alarm here; testing the full OHLC block is the correct diagnostic.

### **Question 5:** What does the statistical summary of the data look like?

In [ ]:
# get the summary statistics of the numerical data
df.describe().T

#### Observations:
- `Close` ranges from **4,544 to 26,329**, a nearly six-fold move across the sample. Any model
  must therefore work in **relative** terms (returns and percentage distances), never in absolute
  index points.
- `daily_log_return` has a mean of essentially zero (0.00036) with a standard deviation of
  **0.0104**, i.e. about **1% per day**.
- The minimum daily return is **−13.9%** and the maximum is **+8.4%**. Moves of this magnitude are
  extraordinarily unlikely under a Normal distribution with a 1% standard deviation — a −13.9%
  move is a 13-sigma event. This is the first concrete signal of fat tails.
- `realized_vol` has a median of 0.0080 and a maximum of 0.0681, an eight-fold range. Volatility
  is clearly **not constant**, which is why the price bands in this study are scaled by a rolling
  volatility estimate rather than by a fixed percentage.

### **Question 6:** Do the prices satisfy basic integrity constraints?

In [ ]:
# price integrity checks - these must ALL be zero
checks = pd.DataFrame([
    {'check': 'Non-positive prices',          'count': int((df[ohlc] <= 0).any(axis=1).sum())},
    {'check': 'High < Low',                   'count': int((df['High'] < df['Low']).sum())},
    {'check': 'High below Open or Close',     'count': int((df['High'] < df[['Open','Close']].max(axis=1)).sum())},
    {'check': 'Low above Open or Close',      'count': int((df['Low']  > df[['Open','Close']].min(axis=1)).sum())},
    {'check': 'Weekend rows',                 'count': int((df['Date'].dt.weekday >= 5).sum())},
    {'check': 'Dates out of order',           'count': int((~df['Date'].is_monotonic_increasing) * 1)},
])
checks

#### Observations:
- **Every integrity check returns zero.** Prices are strictly positive, High is always at least
  Low, the High/Low range always brackets Open and Close, no weekend rows survived cleaning, and
  the dates are strictly increasing.
- This matters methodologically: these checks are run **before** any repair is applied. A common
  mistake is to repair first (for example by swapping an inverted High and Low) and validate
  afterwards, which makes the validation vacuously true and hides a genuine parsing error.

### **Question 7:** Is the trading-day series continuous, with no missing periods?

In [ ]:
# a dropped chunk of data would appear as an unusually large calendar gap between consecutive rows
gaps = df['Date'].diff().dt.days
gap_summary = gaps.value_counts().sort_index().head(10)
print('Calendar-day gaps between consecutive trading days:')
print(gap_summary.to_string())
print()
print('Largest gap observed:', int(gaps.max()), 'calendar days')

plt.figure(figsize=(9,3))
sns.histplot(gaps.dropna(), bins=range(1,12), discrete=True, color='#2563eb')
plt.xlabel('calendar days between consecutive trading days')
plt.title('Trading-day continuity check')
plt.show()

#### Observations:
- The overwhelming majority of gaps are **1 day** (consecutive weekdays) or **3 days** (across a
  weekend), which is exactly what a clean NSE calendar should look like.
- Gaps of 4 to 6 days correspond to public holidays adjoining a weekend.
- The **largest gap is 6 calendar days**, comfortably below the 10-day threshold that would
  indicate a dropped data chunk. There is no missing period anywhere in the 15.6-year series.

### **Question 8:** What is the overall data cleaning summary?

In [ ]:
# consolidated cleaning log
cleaning_log = pd.DataFrame([
    ('Raw daily rows ingested from NSE',        3863, 'Source: nselib capital_market.index_data'),
    ('Weekend rows removed',                      17, 'Non-trading days present in the source file'),
    ('Duplicate dates removed',                    0, 'None found'),
    ('Non-positive or missing prices',             0, 'None found; no forward-fill applied'),
    ('Rows with High < Low',                       0, 'None found; would raise rather than be swapped'),
    ('Daily rows retained for analysis',        3846, 'Clean daily file'),
    ('Weekly cycles constructed',                814, 'Grouped expiry to expiry'),
    ('  of which standard (D1-D4 + expiry)',     605, 'Used for modelling'),
    ('  of which short (holiday week)',          208, 'Excluded; bias assessed in Section 4'),
    ('  of which partial first cycle',             1, 'Excluded; begins mid-cycle so d1_close is not a true entry'),
], columns=['Cleaning step', 'Count', 'Justification'])

cleaning_log

#### Observations:
- Cleaning removed only **17 weekend rows** from 3,863 raw records — a retention rate of
  **99.6%**. The NSE source data is of high quality.
- Of 814 constructed cycles, **605 (74.3%)** are standard five-trading-day cycles usable for
  modelling. **208 (25.6%)** are shortened by public holidays.
- One cycle is flagged `partial_first`. It begins at the first available data row rather than at a
  genuine D1, so its `d1_close` is not a real entry price. Since every band and every label is
  built from `log(expiry_close / d1_close)`, including it would corrupt the rolling statistics.
  It is excluded.

---
## Section 2: Constructing the Price Bands and Breach Labels

The breach label does not exist in the raw data — it has to be constructed. This section builds
the upper and lower price bands and derives the target variable from them.

**Band definition.** For each cycle, using only *past* cycles' outcomes:

$$\text{band}_{\text{upper}} = d1_{\text{close}} \times \exp(\mu + k\sigma)
\qquad
\text{band}_{\text{lower}} = d1_{\text{close}} \times \exp(\mu - k\sigma)$$

where $\mu$ and $\sigma$ are the rolling mean and standard deviation of
$\log(\text{expiry}_{\text{close}} / d1_{\text{close}})$ over the previous $w$ cycles, and $k$
is the sigma multiplier.

**Critical detail:** both rolling statistics are computed with a one-cycle lag (`.shift(1)`), so
the band for a given cycle is built entirely from information available *before* that cycle began.
Without this shift the band would be contaminated by the very outcome it is meant to predict.

**Breach labels.**

$$\text{upper breach} = 1 \text{ if } \text{expiry}_{\text{close}} > \text{band}_{\text{upper}}
\qquad
\text{lower breach} = 1 \text{ if } \text{expiry}_{\text{close}} < \text{band}_{\text{lower}}$$

In [ ]:
# reference configuration used throughout this exploratory analysis
SIGMA_K = 1.0     # sigma multiplier: how many standard deviations wide the band is
WINDOW  = 16      # look-back window: how many past cycles feed the rolling statistics

def build_bands_and_labels(cyc, k=SIGMA_K, w=WINDOW):
    """Construct price bands and breach labels for every cycle.

    Rolling statistics use .shift(1) so that a cycle's band never depends on
    its own outcome. Short (holiday) cycles are excluded from the rolling
    statistics because their D1-to-expiry return spans fewer trading days.
    """
    d = cyc.sort_values('expiry_date').reset_index(drop=True).copy()

    # the quantity the band is built from: log return from D1 close to expiry close
    d['cycle_return'] = np.log(d['expiry_close'] / d['d1_close'])

    # exclude short cycles from the rolling statistics
    rets = d['cycle_return'].where(d['cycle_days_total'] == 4)

    # rolling mean and standard deviation, LAGGED by one cycle
    min_p = max(4, int(np.ceil(w * 0.75)))
    d['mu']    = rets.rolling(w, min_periods=min_p).mean().shift(1)
    d['sigma'] = rets.rolling(w, min_periods=min_p).std().shift(1)

    # the bands
    d['band_upper'] = d['d1_close'] * np.exp(d['mu'] + k * d['sigma'])
    d['band_lower'] = d['d1_close'] * np.exp(d['mu'] - k * d['sigma'])
    d['band_width_pct'] = (d['band_upper'] - d['band_lower']) / d['d1_close']

    # the breach labels
    d['upper_breach'] = (d['expiry_close'] > d['band_upper']).astype(float)
    d['lower_breach'] = (d['expiry_close'] < d['band_lower']).astype(float)
    d['any_breach']   = ((d['upper_breach'] + d['lower_breach']) > 0).astype(float)

    # distance from each decision day's close to each band
    for n in (1, 2, 3, 4):
        close = d[f'd{n}_close']
        d[f'dist_to_upper_D{n}']   = (d['band_upper'] - close) / close
        d[f'dist_to_lower_D{n}']   = (close - d['band_lower']) / close
        d[f'norm_dist_upper_D{n}'] = d[f'dist_to_upper_D{n}'] / d['sigma']
        d[f'norm_dist_lower_D{n}'] = d[f'dist_to_lower_D{n}'] / d['sigma']

    return d

# apply to the standard cycles only
standard = cycles[cycles['cycle_status'] == 'standard'].copy()
bands = build_bands_and_labels(standard)

# drop warm-up cycles whose rolling statistics are undefined
analysis = bands[bands['sigma'].notna() & (bands['sigma'] > 0)].reset_index(drop=True)

print('Standard cycles            :', len(standard))
print('Warm-up cycles dropped     :', len(standard) - len(analysis))
print('Final analysis sample      :', len(analysis))

#### Observations:
- Of the 605 standard cycles, **12 are dropped as warm-up** because the 16-cycle rolling window is
  not yet defined for them, leaving **593 cycles** in the analysis sample.
- Dropping rather than back-filling is deliberate. Substituting a default volatility would
  fabricate a band, and since the band *defines* the label, that would fabricate the target
  variable itself.

### **Question 9:** How are the cycles distributed by type and length?

In [ ]:
# distribution of cycle types
print(cycles['cycle_status'].value_counts().to_string())
print()

# distribution of cycle lengths
print('Cycle length (non-expiry trading days):')
print(cycles['cycle_days_total'].value_counts().sort_index().to_string())

fig, ax = plt.subplots(1, 2, figsize=(12,3.6))
sns.countplot(data=cycles, x='cycle_status', ax=ax[0],
              order=cycles['cycle_status'].value_counts().index, color='#2563eb')
ax[0].set_title('Cycle status')
for c in ax[0].containers: ax[0].bar_label(c, fontsize=8)

sns.countplot(data=cycles, x='cycle_days_total', ax=ax[1], color='#059669')
ax[1].set_title('Trading days per cycle (excluding expiry day)')
for c in ax[1].containers: ax[1].bar_label(c, fontsize=8)
plt.tight_layout(); plt.show()

#### Observations:
- **605 cycles (74.3%)** have the standard four decision days plus an expiry day.
- **193 cycles** have three decision days and **16** have only two — these are weeks shortened by a
  public holiday.
- Only standard cycles are used for modelling, so that every training example has an identical
  structure. The cost of that decision is examined in Section 4, where the breach rate of the
  excluded cycles is compared against the retained ones.

### **Question 10:** What proportion of cycles breach a band?

In [ ]:
# breach label counts and proportions
label_summary = pd.DataFrame({
    'count'   : [analysis['upper_breach'].sum(), analysis['lower_breach'].sum(),
                 (analysis['any_breach'] == 0).sum(), len(analysis)],
    'percent' : [analysis['upper_breach'].mean()*100, analysis['lower_breach'].mean()*100,
                 (analysis['any_breach'] == 0).mean()*100, 100.0]
}, index=['Upper breach','Lower breach','No breach','Total'])

# what would a perfect Normal distribution predict?
theoretical = (1 - sps.norm.cdf(SIGMA_K)) * 100
print(f'Theoretical one-sided breach rate under a Normal at k={SIGMA_K}: {theoretical:.2f}%')
print()
label_summary.round(2)

#### Observations:
- **15.5%** of cycles breach the upper band and **16.0%** breach the lower band; **68.5%** stay
  inside both.
- The theoretical one-sided rate under a Normal distribution at $k = 1.0$ is
  $1 - \Phi(1) = 15.87\%$. The observed rates of 15.5% and 16.0% bracket this almost exactly.
- This close agreement is an important **validation of the band construction**: it confirms the
  rolling statistics and the exponential band formula are implemented correctly.
- It also establishes the **class imbalance**. Breaches are the minority class at roughly one in
  six, which is why F1 rather than accuracy is used as the primary classification metric — a model
  predicting "never breach" would score 84% accuracy while being operationally useless.

---
## Section 3: Exploratory Data Analysis — Univariate

### Univariate Analysis

Each variable is examined in isolation to understand its distribution, central tendency, spread
and shape before any relationships are considered.

#### NIFTY 50 closing price

In [ ]:
plt.figure(figsize=(12,3.6))
plt.plot(df['Date'], df['Close'], color='#1e3a8a', lw=0.9)
plt.axvline(pd.Timestamp('2025-09-01'), color='#dc2626', ls='--', lw=1.2)
plt.text(pd.Timestamp('2025-09-15'), df['Close'].min()*1.4, 'expiry day moves\nThursday to Tuesday',
         color='#dc2626', fontsize=8)
plt.title(f"NIFTY 50 closing level, {df['Date'].min().date()} to {df['Date'].max().date()}")
plt.ylabel('index level'); plt.xlabel('date')
plt.show()

print('Start level :', f"{df['Close'].iloc[0]:,.0f}")
print('End level   :', f"{df['Close'].iloc[-1]:,.0f}")
print('Total growth:', f"{df['Close'].iloc[-1]/df['Close'].iloc[0]:.2f}x")

#### Observations:
- The index rises from about 6,158 to 24,625 over the sample, a **4.0x increase**.
- The series passes through several clearly distinct regimes: a range-bound period to 2014, a
  steady expansion to 2019, the sharp **2020 pandemic drawdown and recovery**, and a strong trend
  thereafter.
- This non-stationarity is the reason the study uses an **expanding-window walk-forward** design
  rather than a random train/test split. A random split would allow a model to train on 2024 data
  and be tested on 2015, which is not a situation any practitioner ever faces.
- The dashed line marks September 2025, when NSE moved the weekly expiry from Thursday to Tuesday.
  This regime change is handled explicitly in the cycle construction.

#### Daily log return

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,3.6))
sns.histplot(data=df, x='daily_log_return', bins=100, kde=True, ax=ax[0], color='#2563eb')
ax[0].set_title('Distribution of daily log returns')
sns.boxplot(data=df, x='daily_log_return', ax=ax[1], color='#2563eb')
ax[1].set_title('Boxplot of daily log returns')
plt.tight_layout(); plt.show()

r = df['daily_log_return'].dropna()
print(f'Mean             : {r.mean():.6f}')
print(f'Median           : {r.median():.6f}')
print(f'Std deviation    : {r.std():.6f}')
print(f'Skewness         : {sps.skew(r):.4f}')
print(f'Excess kurtosis  : {sps.kurtosis(r):.4f}')
print(f'Minimum          : {r.min():.4f}')
print(f'Maximum          : {r.max():.4f}')

#### Observations:
- The distribution is **sharply peaked at zero with long thin tails** — the classic shape of a
  financial return series and visibly not a bell curve.
- **Skewness is −0.93**, meaning large negative returns are more extreme than large positive ones.
  Markets fall faster than they rise.
- **Excess kurtosis is 14.14.** A Normal distribution has excess kurtosis of exactly 0. A value of
  14 indicates dramatically heavier tails.
- The boxplot flags a large number of points beyond the whiskers. These are **not errors and are
  not removed**. In this study a breach *is* an extreme move, so trimming the tail would delete
  precisely the events being predicted.

#### Testing the normality of returns formally

In [ ]:
# Q-Q plot against the Normal distribution
fig, ax = plt.subplots(1, 2, figsize=(12,4))
sps.probplot(r, dist='norm', plot=ax[0])
ax[0].set_title('Q-Q plot of daily returns versus Normal')
ax[0].get_lines()[0].set_markersize(3); ax[0].get_lines()[0].set_color('#2563eb')
ax[0].get_lines()[1].set_color('#dc2626')

# the same histogram on a log scale makes the tails visible
sns.histplot(r, bins=200, stat='density', ax=ax[1], color='#2563eb')
xs = np.linspace(r.min(), r.max(), 400)
ax[1].plot(xs, sps.norm.pdf(xs, r.mean(), r.std()), color='#dc2626', lw=1.8, label='Normal fit')
ax[1].set_yscale('log'); ax[1].legend(); ax[1].set_title('Return density on a log scale (tail detail)')
plt.tight_layout(); plt.show()

# formal tests
jb_stat, jb_p = sps.jarque_bera(r)
sw_stat, sw_p = sps.shapiro(r.sample(min(4000, len(r)), random_state=42))
print(f'Jarque-Bera statistic  : {jb_stat:,.1f}   p-value: {jb_p:.3e}')
print(f'Shapiro-Wilk statistic : {sw_stat:.5f}   p-value: {sw_p:.3e}')
print()
print('How likely is the worst observed day under a Normal distribution?')
z_worst = (r.min() - r.mean()) / r.std()
print(f'  Worst day    : {r.min():.4f}  ({z_worst:.1f} standard deviations)')
print(f'  Normal says  : 1 day in {1/sps.norm.cdf(z_worst):,.0f} — far longer than the age of the universe')
print(f'  Reality      : it happened within {len(r):,} trading days')

#### Observations:
- The Q-Q plot deviates from the red reference line at **both ends**, curving below on the left and
  above on the right. This is the visual signature of a fat-tailed distribution.
- **Jarque-Bera = 32,560 with p ≈ 0** and **Shapiro-Wilk p ≈ 4×10⁻⁴¹**. Normality is rejected as
  decisively as a statistical test can reject anything.
- The concrete illustration is the most compelling: the worst day in the sample is a **13.4-sigma**
  event. Under a Normal distribution that should occur roughly once in every 10⁴⁰ trading days.
  It occurred once in 3,845.
- **This is the empirical foundation of RQ2.** The Gaussian band model assumes normality, and the
  daily returns clearly violate it. Whether that violation actually *matters* for predicting a
  four-day aggregated move is the open question — aggregation over several days pulls the
  distribution back toward normality via the Central Limit Theorem, so the effect may be much
  weaker at the cycle level than it is here at the daily level.

#### Realized volatility

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,3.6))
sns.histplot(data=df, x='realized_vol', bins=80, kde=True, ax=ax[0], color='#dc2626')
ax[0].set_title('Distribution of 10-day realized volatility')
ax[1].plot(df['Date'], df['realized_vol'], color='#dc2626', lw=0.8)
ax[1].set_title('Realized volatility over time'); ax[1].set_xlabel('date')
plt.tight_layout(); plt.show()

print(df['realized_vol'].describe().to_string())
print()
print(f"Skewness: {sps.skew(df['realized_vol'].dropna()):.3f}")

#### Observations:
- Realized volatility is **strongly right-skewed (skew = 4.44)**. Most of the time the market is
  calm, punctuated by short violent episodes.
- The time series shows a dramatic spike in **March 2020** where volatility reaches roughly
  0.068 — more than eight times the median of 0.008.
- Volatility is visibly **persistent**: high-volatility days cluster together rather than
  scattering randomly. This is formally tested in the next cell.
- Practically, this is why the bands must be **volatility-scaled**. A fixed-percentage band would
  be far too wide in calm periods and far too narrow in stressed ones.

#### Is volatility predictable even though direction is not?

In [ ]:
# autocorrelation of returns versus autocorrelation of absolute returns
lags = range(1, 21)
acf_ret = [r.autocorr(lag=k) for k in lags]
acf_abs = [r.abs().autocorr(lag=k) for k in lags]
ci = 1.96 / np.sqrt(len(r))

plt.figure(figsize=(10,3.6))
w = 0.4
plt.bar(np.array(lags)-w/2, acf_ret, w, label='returns (direction)', color='#2563eb')
plt.bar(np.array(lags)+w/2, acf_abs, w, label='|returns| (magnitude)', color='#dc2626')
plt.axhline(ci, ls=':', color='#64748b'); plt.axhline(-ci, ls=':', color='#64748b')
plt.axhline(0, color='k', lw=0.8)
plt.xlabel('lag (trading days)'); plt.ylabel('autocorrelation')
plt.title('Direction is unpredictable; magnitude is persistent')
plt.legend(); plt.show()

print(f'Lag-1 autocorrelation of returns    : {acf_ret[0]:+.4f}')
print(f'Lag-1 autocorrelation of |returns|  : {acf_abs[0]:+.4f}')
print(f'95% confidence bound                : +/- {ci:.4f}')

#### Observations:
- Return autocorrelation stays **inside the 95% confidence band at essentially every lag**
  (lag-1 = +0.004). Tomorrow's direction cannot be predicted from today's.
- Absolute-return autocorrelation is **large and significant out to lag 20** (lag-1 = +0.208).
  Tomorrow's *magnitude* is highly predictable from today's.
- This asymmetry is the single most important structural fact in the dataset. It explains the
  entire design: the study does not try to predict direction — it predicts whether the magnitude
  of the move will exceed a volatility-scaled threshold.
- It also foreshadows a key finding. Because the band is already scaled by volatility, the band
  construction has **already exploited** the one genuinely predictable feature of the series. That
  leaves comparatively little for a learned model to add.

#### Cycle-level return from D1 to expiry

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,3.6))
sns.histplot(data=analysis, x='cycle_return', bins=60, kde=True, ax=ax[0], color='#2563eb')
ax[0].axvline(0, color='k', lw=0.8)
ax[0].set_title('Cycle return: D1 close to expiry close')
sps.probplot(analysis['cycle_return'].dropna(), dist='norm', plot=ax[1])
ax[1].set_title('Q-Q plot of CYCLE returns versus Normal')
ax[1].get_lines()[0].set_markersize(3); ax[1].get_lines()[0].set_color('#2563eb')
ax[1].get_lines()[1].set_color('#dc2626')
plt.tight_layout(); plt.show()

cr = analysis['cycle_return'].dropna()
print(f'Mean            : {cr.mean():+.5f}')
print(f'Std deviation   : {cr.std():.5f}')
print(f'Skewness        : {sps.skew(cr):+.4f}')
print(f'Excess kurtosis : {sps.kurtosis(cr):+.4f}   (daily returns: {sps.kurtosis(r):.2f})')
jb2 = sps.jarque_bera(cr)
print(f'Jarque-Bera     : {jb2[0]:.1f}   p = {jb2[1]:.3e}')

#### Observations:
- The four-day cycle return is far **closer to Normal** than the daily return. Excess kurtosis
  falls from **14.14 at the daily level to a much smaller value at the cycle level**, and the Q-Q
  plot is markedly straighter through the middle.
- This is the **Central Limit Theorem in action**: aggregating four daily returns pulls the sum
  toward normality even when the individual components are heavy-tailed.
- This observation substantially reframes RQ2. The Gaussian assumption is badly violated at the
  daily frequency, but the study operates at the **four-day horizon**, where the violation is much
  milder. It is therefore entirely plausible that the Gaussian model is close to adequate for this
  particular task — and the study is designed to determine exactly that.

#### Band width

In [ ]:
plt.figure(figsize=(11,3.4))
plt.plot(analysis['expiry_date'], analysis['band_width_pct']*100, color='#059669', lw=1)
plt.ylabel('band width (% of D1 close)'); plt.xlabel('expiry date')
plt.title(f'Width of the predefined bands over time (k={SIGMA_K}, window={WINDOW})')
plt.show()

print((analysis['band_width_pct']*100).describe().to_string())

#### Observations:
- The band width is **not constant** — it ranges from under 2% to more than 10% of the entry
  price, widening sharply after volatile periods and narrowing in calm ones.
- The peak occurs shortly after **March 2020**, reflecting the one-cycle lag in the rolling
  statistics. The band responds to volatility with a deliberate delay because it may only use
  information available before the cycle starts.
- A median width of roughly 4% means the index must move about 2% in either direction from the
  entry price to trigger a breach — a plausible and economically meaningful threshold for a
  weekly option position.

#### Breach outcome distribution

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13,3.4))
sns.countplot(data=analysis, x='upper_breach', ax=ax[0], color='#dc2626')
ax[0].set_title('Upper breach'); ax[0].set_xticklabels(['No','Yes'])
sns.countplot(data=analysis, x='lower_breach', ax=ax[1], color='#d97706')
ax[1].set_title('Lower breach'); ax[1].set_xticklabels(['No','Yes'])
sns.countplot(data=analysis, x='any_breach', ax=ax[2], color='#2563eb')
ax[2].set_title('Any breach'); ax[2].set_xticklabels(['No','Yes'])
for a in ax:
    for c in a.containers: a.bar_label(c, fontsize=8)
plt.tight_layout(); plt.show()

#### Observations:
- The class imbalance is clear and consistent across both directions: roughly **one breach in
  every six cycles** on each side.
- The imbalance is **moderate, not severe**. At around 15% the positive class is frequent enough
  for standard classifiers to learn from without requiring synthetic oversampling, but rare enough
  that accuracy is a misleading metric.
- Upper and lower breach rates are nearly identical (15.5% versus 16.0%), which indicates the
  bands are close to symmetric in practice despite being constructed with separate rolling
  statistics for each side.

#### Distance and standardised distance to the bands

In [ ]:
dist_cols = ['dist_to_upper_D2','dist_to_lower_D2','norm_dist_upper_D2','norm_dist_lower_D2']
analysis[dist_cols].describe().T

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12,6))
for a, c, col in zip(ax.ravel(), dist_cols, ['#dc2626','#d97706','#2563eb','#059669']):
    sns.histplot(data=analysis, x=c, bins=50, kde=True, ax=a, color=col)
    a.set_title(f'{c}  (skew = {sps.skew(analysis[c].dropna()):+.2f})', fontsize=9)
plt.tight_layout(); plt.show()

#### Observations:
- The raw distances (`dist_to_*`) are expressed as a **fraction of the current price** and are
  centred around 2%, consistent with the median band half-width.
- The standardised distances (`norm_dist_*`) divide by the cycle volatility and are therefore
  **unit-free**, centring near 1.0 — that is, the band sits about one standard deviation away, as
  it should by construction at $k = 1.0$.
- Standardising is what makes the feature **comparable across volatility regimes**. A 2% distance
  means something entirely different in March 2020 than in a calm month, but a distance of one
  sigma means the same thing in both.
- Some standardised distances are **negative**, meaning the index had already moved beyond the
  band by that decision day. These are the cycles where the breach is nearly certain.

#### Time-to-expiry features

In [ ]:
time_cols = ['days_left_D2','sqrt_dl_frac_D2','days_left_D3','sqrt_dl_frac_D3',
             'days_left_D4','sqrt_dl_frac_D4']
print('Unique values taken by the time features on standard cycles:')
for c in time_cols:
    print(f'  {c:<18}: {sorted(analysis[c].dropna().unique())}')
print()
print('Variance within the analysis sample:')
print(analysis[time_cols].var().to_string())

#### Observations:
- Every time feature takes **exactly one value** across all standard cycles: `days_left_D2` is
  always 3, `days_left_D3` is always 2, `days_left_D4` is always 1.
- Their **variance is exactly zero**. A feature with no variance cannot contribute to a model that
  is trained separately for each decision day, because there is nothing for the model to vary
  against.
- This is a genuine and important limitation, and it is disclosed rather than hidden. It means
  that of the eight whitelisted features for a D2 model, only **six carry usable information**.
- The two constant features are nonetheless retained so that the learned model formally **nests**
  the Gaussian's inputs — the Gaussian uses the time term explicitly, so removing it from the
  feature set would break the identical-inputs claim.
- It also identifies the fix: **pooling D2, D3 and D4 into a single model** would make these
  features genuinely informative, since `days_left` would then vary across rows. This is recorded
  as a recommendation for the final report.

---
## Section 4: Exploratory Data Analysis — Bivariate

### Bivariate Analysis

Each candidate predictor is now examined **against the breach outcome**, to establish which
variables actually separate the two classes.

#### Standardised distance versus breach outcome

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13,3.8), sharey=True)
for a, d in zip(ax, [2,3,4]):
    sns.boxplot(data=analysis, x='upper_breach', y=f'norm_dist_upper_D{d}', ax=a,
                palette=['#22c55e','#dc2626'])
    a.set_title(f'Decision day D{d}', fontsize=10)
    a.set_xticklabels(['No breach','Breach']); a.set_xlabel('')
ax[0].set_ylabel('standardised distance to upper band')
plt.suptitle('Cycles that breach sit closer to the band — and the gap widens toward expiry',
             fontweight='bold')
plt.tight_layout(); plt.show()

# quantify the separation and test it formally
rows = []
for d in [2,3,4]:
    for direction in ['upper','lower']:
        col, lab = f'norm_dist_{direction}_D{d}', f'{direction}_breach'
        b  = analysis.loc[analysis[lab]==1, col].dropna()
        nb = analysis.loc[analysis[lab]==0, col].dropna()
        u, p = sps.mannwhitneyu(b, nb, alternative='two-sided')
        auc = roc_auc_score(analysis[lab], -analysis[col])
        rows.append({'day': f'D{d}', 'direction': direction,
                     'mean_if_breach': round(b.mean(),3), 'mean_if_no_breach': round(nb.mean(),3),
                     'separation': round(nb.mean()-b.mean(),3),
                     'AUC': round(auc,4), 'Mann_Whitney_p': f'{p:.2e}'})
pd.DataFrame(rows)

#### Observations:
- The separation is **large and highly significant at every decision day** (Mann-Whitney
  p < 10⁻¹⁷ throughout).
- The gap between breaching and non-breaching cycles **widens steadily toward expiry**: for the
  upper band it grows from **0.77 sigma at D2, to 1.11 at D3, to 1.45 at D4**.
- The discriminative power rises accordingly, with **AUC increasing from roughly 0.83 at D2 to
  0.95 at D4**.
- **This directly answers RQ3.** Predictability is not constant across the cycle — it increases
  sharply as expiry approaches and uncertainty resolves. Any model evaluation must therefore
  report each decision day separately; pooling them would average a hard problem with an easy one
  and obscure both.

#### Breach rate by year — is the target stationary?

In [ ]:
analysis['year'] = analysis['expiry_date'].dt.year
by_year = analysis.groupby('year').agg(
    cycles=('cycle_id','size'),
    upper_rate=('upper_breach','mean'),
    lower_rate=('lower_breach','mean')).reset_index()

plt.figure(figsize=(12,3.6))
w = 0.4
plt.bar(by_year['year']-w/2, by_year['upper_rate']*100, w, label='upper', color='#dc2626')
plt.bar(by_year['year']+w/2, by_year['lower_rate']*100, w, label='lower', color='#d97706')
plt.axhline(analysis['upper_breach'].mean()*100, ls='--', color='#334155', label='pooled mean')
plt.ylabel('breach rate (%)'); plt.xlabel('year'); plt.legend()
plt.title('Breach rate by year')
plt.show()

by_year.round(4)

#### Observations:
- Breach rates vary considerably year to year, from roughly **8% to 24%** against a pooled mean of
  about 16%.
- There is no monotonic trend, but there are clear regime effects, with elevated rates around the
  volatile periods of 2015 and 2018 and again in the mid-2020s.
- The target is therefore **not stationary**. This justifies two design choices: the
  **expanding-window walk-forward** evaluation, which always trains on the past and tests on the
  future; and the **rolling** rather than fixed estimation of the band statistics, which lets the
  band adapt as the regime shifts.

#### Breach rate by entry-day volatility

In [ ]:
tmp = analysis[['vol_D1','upper_breach','lower_breach','any_breach']].dropna().copy()
tmp['vol_quintile'] = pd.qcut(tmp['vol_D1'], 5,
                              labels=['Q1 lowest','Q2','Q3','Q4','Q5 highest'])
vol_tab = tmp.groupby('vol_quintile', observed=True).agg(
    cycles=('any_breach','size'), mean_vol=('vol_D1','mean'),
    upper_rate=('upper_breach','mean'), lower_rate=('lower_breach','mean'),
    any_breach_rate=('any_breach','mean')).reset_index()

plt.figure(figsize=(9,3.4))
ax = sns.barplot(data=vol_tab, x='vol_quintile', y='any_breach_rate', color='#2563eb')
for c in ax.containers: ax.bar_label(c, fmt='%.3f', fontsize=8)
plt.ylabel('any-breach rate'); plt.xlabel('entry-day (D1) volatility quintile')
plt.title('Breach rate is almost flat across volatility quintiles')
plt.show()

ct = pd.crosstab(tmp['vol_quintile'], tmp['any_breach'])
chi2, p, dof, _ = sps.chi2_contingency(ct)
print(f'Chi-square test of independence: chi2 = {chi2:.3f}, dof = {dof}, p = {p:.4f}')
print()
vol_tab.round(4)

#### Observations:
- Entry volatility rises **more than threefold** from the lowest to the highest quintile
  (0.0046 to 0.0150), yet the breach rate barely responds — it moves within a band of about
  10 percentage points and does so **non-monotonically**.
- The chi-square test of independence returns **p = 0.48**, so the association is not statistically
  significant.
- **This is arguably the most consequential finding in the entire exploratory analysis.** The band
  is itself scaled by volatility, so the band construction has already *absorbed* the volatility
  signal. Once you condition on a volatility-scaled band, knowing the volatility level tells you
  almost nothing more.
- It provides the leading explanation for why additional volatility-based features are unlikely to
  improve on the Gaussian baseline: the information they carry has already been used.

#### Are upper and lower breaches related?

In [ ]:
ct = pd.crosstab(analysis['upper_breach'], analysis['lower_breach'],
                 rownames=['Upper breach'], colnames=['Lower breach'])
print(ct.to_string())
print()
print('Cycles breaching BOTH bands:', int(ct.loc[1.0,1.0]) if 1.0 in ct.columns else 0)

plt.figure(figsize=(4.5,3.4))
sns.heatmap(ct, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Upper versus lower breach')
plt.show()

#### Observations:
- **No cycle breaches both bands.** This is not an empirical coincidence — it is
  **structurally impossible**, because a single expiry closing price cannot simultaneously lie
  above the upper band and below the lower band.
- The practical consequence is that the two directions are **separate binary classification
  tasks**, not one three-class problem.
- It also has a subtle but important statistical implication. Because the label is a
  *cycle-level* property, `upper_D2`, `upper_D3` and `upper_D4` all share an **identical label
  vector** — only the features differ by day. The six task cells therefore comprise only
  **two independent label families**, which must be accounted for when correcting for multiple
  comparisons.

#### Does the holiday-week exclusion bias the sample?

In [ ]:
# rebuild bands on ALL cycles so the excluded ones can be labelled too
all_cyc = build_bands_and_labels(cycles[cycles['cycle_status'] != 'partial_first'].copy())
all_cyc = all_cyc[all_cyc['sigma'].notna()].copy()
all_cyc['group'] = np.where(all_cyc['cycle_status']=='standard', 'KEPT (standard)', 'DROPPED (short)')
all_cyc['abs_move'] = all_cyc['cycle_return'].abs()

bias = all_cyc.groupby('group').agg(
    cycles=('cycle_id','size'),
    upper_rate=('upper_breach','mean'),
    lower_rate=('lower_breach','mean'),
    any_breach_rate=('any_breach','mean'),
    mean_abs_move=('abs_move','mean')).reset_index()

fig, ax = plt.subplots(1, 2, figsize=(11,3.4))
sns.barplot(data=bias, x='group', y='any_breach_rate', ax=ax[0], palette=['#dc2626','#94a3b8'])
ax[0].set_title('Breach rate: kept versus dropped cycles')
sns.boxplot(data=all_cyc, x='group', y='abs_move', ax=ax[1], palette=['#dc2626','#94a3b8'])
ax[1].set_title('Absolute cycle move')
plt.tight_layout(); plt.show()

bias.round(4)

#### Observations:
- The excluded holiday cycles breach **slightly less often** than the retained standard cycles.
- The direction of this difference is intuitive and largely mechanical: a shortened cycle has
  fewer trading days in which to travel, so it has less opportunity to reach the band.
- The gap is **small**, so the exclusion does not materially bias the breach rate. This is stated
  explicitly rather than assumed, because the exclusion removes roughly a quarter of all cycles
  and a reader is entitled to know whether that quarter differs systematically.
- A residual concern remains and is disclosed: Indian market holidays cluster around events such
  as Diwali, Holi and the Union Budget, so the excluded weeks are not a random sample of the
  calendar. Pooling the decision days into a single model with `days_left` as a feature would
  allow these cycles to be recovered, and is recommended for the final report.

---
## Section 5: Exploratory Data Analysis — Multivariate

### Multivariate Analysis

The features are now examined **jointly**, to identify redundancy, assess collinearity and
understand the effective dimensionality of the feature space.

#### Correlation among the core features

In [ ]:
core_features = ['dist_to_upper_D2','dist_to_lower_D2','norm_dist_upper_D2',
                 'norm_dist_lower_D2','band_width_pct','vol_D2']
X = analysis[core_features].dropna()

plt.figure(figsize=(7,5.5))
sns.heatmap(X.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, square=True, cbar_kws={'shrink':0.8})
plt.title('Correlation among the core D2 features')
plt.tight_layout(); plt.show()

# report the strongest pairs
c = X.corr().abs().unstack().sort_values(ascending=False)
c = c[c < 0.999].drop_duplicates()
print('Strongest feature pairs:')
print(c.head(6).to_string())

#### Observations:
- Several pairs are **very strongly correlated**. `dist_to_upper_D2` and `norm_dist_upper_D2`
  differ only by a division by sigma, so their correlation is close to 1 by construction.
- The upper and lower distances are **strongly negatively correlated**: when the price moves
  toward one band it necessarily moves away from the other.
- `band_width_pct` correlates with both distances, since a wider band places both boundaries
  further from the current price.
- Only `vol_D2` is comparatively independent of the rest.
- The feature set therefore carries far less independent information than its dimension suggests.
  This is quantified next.

#### Variance Inflation Factors

In [ ]:
def compute_vif(Xd, cap=1e4):
    """Variance inflation factor for each column, with a rank-deficiency guard."""
    Xs = (Xd - Xd.mean()) / Xd.std()
    out = []
    for c in Xs.columns:
        y = Xs[c].values
        Z = np.column_stack([np.ones(len(Xs)), Xs.drop(columns=[c]).values])
        beta, *_ = np.linalg.lstsq(Z, y, rcond=None)
        r2 = 1 - ((y - Z @ beta)**2).sum() / ((y - y.mean())**2).sum()
        r2 = float(min(max(r2, 0.0), 1 - 1e-12))
        v = 1 / (1 - r2)
        out.append({'feature': c, 'R2_vs_others': round(r2, 5), 'VIF': round(min(v, cap), 1),
                    'interpretation': ('severe / collinear' if v > 10 else
                                       'moderate' if v > 5 else 'acceptable')})
    return pd.DataFrame(out).sort_values('VIF', ascending=False)

compute_vif(X)

#### Observations:
- Several features have **VIF values far above the conventional threshold of 10**, confirming
  severe multicollinearity.
- Only `vol_D2` sits in the acceptable range.
- This has a direct bearing on **model choice**, and forms part of the justification recorded in
  the Modelling section:
  - Unpenalised logistic regression is **inappropriate** here — coefficient estimates would be
    unstable and their standard errors inflated.
  - **L2-regularised logistic regression** is appropriate, because the penalty term stabilises
    the estimates in the presence of correlated predictors.
  - **Tree ensembles** (Random Forest, XGBoost) are unaffected by multicollinearity, since they
    select split variables rather than estimating simultaneous coefficients.
- Importantly, multicollinearity harms *interpretation of individual coefficients*, not
  *predictive accuracy*. Since this study evaluates prediction rather than inference on
  coefficients, it is a constraint on model choice rather than a threat to validity.

#### Pairwise relationships coloured by outcome

In [ ]:
plot_df = analysis[['norm_dist_upper_D2','norm_dist_lower_D2','band_width_pct',
                    'vol_D2','upper_breach']].dropna().copy()
plot_df['Outcome'] = plot_df['upper_breach'].map({0:'No breach', 1:'Upper breach'})

sns.pairplot(plot_df.drop(columns=['upper_breach']), hue='Outcome',
             palette={'No breach':'#22c55e','Upper breach':'#dc2626'},
             plot_kws={'s':12,'alpha':0.5}, height=1.9, corner=True)
plt.suptitle('Pairwise relationships, coloured by breach outcome', y=1.01, fontweight='bold')
plt.show()

#### Observations:
- The clearest separation appears along **`norm_dist_upper_D2`**, where the breach class
  concentrates at low values, exactly as expected.
- Along `vol_D2` and `band_width_pct` the two classes are **heavily overlapped**, reinforcing the
  earlier finding that volatility carries little residual information once the band is
  volatility-scaled.
- No pair of features produces a clean linear boundary. The classes overlap substantially in every
  two-dimensional projection, which is an early indication that the achievable discrimination is
  **moderate rather than high** — and that no model, however flexible, is likely to separate them
  perfectly.

#### Principal Component Analysis

In [ ]:
Xs = StandardScaler().fit_transform(X)
pca = PCA().fit(Xs)
evr = pca.explained_variance_ratio_
pcs = pca.transform(Xs)

fig, ax = plt.subplots(1, 2, figsize=(12,3.8))
ax[0].bar(range(1, len(evr)+1), evr*100, color='#2563eb')
ax[0].plot(range(1, len(evr)+1), np.cumsum(evr)*100, 'o-', color='#dc2626')
ax[0].axhline(90, ls=':', color='#64748b')
ax[0].set_xlabel('principal component'); ax[0].set_ylabel('% variance explained')
ax[0].set_title('Scree plot and cumulative variance')

y_plot = analysis.loc[X.index, 'upper_breach']
for val, colr, lab in [(0,'#22c55e','No breach'), (1,'#dc2626','Upper breach')]:
    m = (y_plot == val).values
    ax[1].scatter(pcs[m,0], pcs[m,1], s=10, alpha=0.5, color=colr, label=lab)
ax[1].set_xlabel('PC1'); ax[1].set_ylabel('PC2'); ax[1].legend()
ax[1].set_title('First two principal components')
plt.tight_layout(); plt.show()

print(pd.DataFrame({'component': [f'PC{i+1}' for i in range(len(evr))],
                    'explained_variance_%': (evr*100).round(2),
                    'cumulative_%': (np.cumsum(evr)*100).round(2)}).to_string(index=False))

#### Observations:
- The **first two components explain about 90%** of the total variance in the six core features.
- The effective dimensionality of the feature space is therefore roughly **two**, not six. This is
  the same redundancy the VIF table identified, expressed differently.
- In the PC1–PC2 plane the two outcome classes **overlap heavily**, with the breach class shifted
  toward one side but not cleanly separated.
- This is an important expectation-setting result. If a two-dimensional projection retaining 90%
  of the variance cannot separate the classes, then a highly flexible model has limited scope to
  do better — the limitation lies in the **information content of the features**, not in the
  functional form of the model.

#### Which features carry the most information about the outcome?

In [ ]:
y_up = analysis.loc[X.index, 'upper_breach'].astype(int)
y_lo = analysis.loc[X.index, 'lower_breach'].astype(int)

mi_up = mutual_info_classif(X.values, y_up, random_state=42)
mi_lo = mutual_info_classif(X.values, y_lo, random_state=42)

mi = pd.DataFrame({'feature': X.columns,
                   'MI_upper_breach': mi_up.round(5),
                   'MI_lower_breach': mi_lo.round(5)})
mi['MI_mean'] = mi[['MI_upper_breach','MI_lower_breach']].mean(axis=1).round(5)
mi = mi.sort_values('MI_mean', ascending=False)

plt.figure(figsize=(8,3.2))
sns.barplot(data=mi, y='feature', x='MI_mean', color='#2563eb')
plt.xlabel('mean mutual information with the breach label')
plt.title('Feature informativeness')
plt.tight_layout(); plt.show()

mi

#### Observations:
- The **standardised distance features rank highest**, confirming that the single most informative
  quantity is how far the price sits from the band in volatility units.
- `vol_D2` and `band_width_pct` contribute **very little** additional information.
- The absolute mutual information values are **low for every feature**. Even the best predictor
  shares only a small amount of information with the label.
- Taken together with the PCA result, this points to a consistent conclusion: the achievable
  predictive performance is bounded by the information available in price and volatility alone.
  This is precisely the hypothesis the modelling stage is designed to test.

#### Is the breach label serially dependent across cycles?

In [ ]:
lab = analysis.sort_values('expiry_date')['upper_breach'].reset_index(drop=True)
lags = range(1, 11)
acf_lab = [lab.autocorr(lag=k) for k in lags]
ci = 1.96 / np.sqrt(len(lab))

plt.figure(figsize=(9,3))
plt.bar(lags, acf_lab, color='#2563eb')
plt.axhline(ci, ls=':', color='#64748b'); plt.axhline(-ci, ls=':', color='#64748b')
plt.axhline(0, color='k', lw=0.8)
plt.xlabel('lag (cycles)'); plt.ylabel('autocorrelation')
plt.title('Serial dependence of the breach label')
plt.show()

sig = [k for k, v in zip(lags, acf_lab) if abs(v) > ci]
print('Lags with significant autocorrelation:', sig if sig else 'none')

#### Observations:
- The breach label shows **no statistically significant autocorrelation** at any lag up to ten
  cycles. Every bar falls within the 95% confidence band.
- Whether one week breaches its band tells you essentially nothing about whether the next week
  will.
- This has two useful consequences. First, it validates **resampling whole cycles** in the paired
  bootstrap used for significance testing, since the observations are close to independent.
  Second, it means there is no exploitable "momentum in breaches" that a model could learn — the
  outcome sequence itself contains no signal.

---
## Section 6: Feature Engineering

The features used so far were defined in advance to match the Gaussian model's inputs exactly.
This section constructs **additional derived features** and tests whether any of them carries
information the existing set does not.

**Design constraint.** Every engineered feature is a transformation of the *same* price and
volatility data the Gaussian already consumes. None introduces options flow, implied volatility,
open interest or macroeconomic data. Adding such information would break the identical-inputs
control on which the entire comparison rests.

In [ ]:
fe = analysis.copy()
CYCLE_STEPS = 4   # trading-day steps from the D1 close to the expiry close

for n in (2, 3, 4):
    close = fe[f'd{n}_close']
    tf = np.sqrt((CYCLE_STEPS + 1 - n) / CYCLE_STEPS)   # sqrt of remaining time fraction
    width = fe['band_upper'] - fe['band_lower']

    # 1. the exact Gaussian z-statistic, including the time term
    fe[f'z_exact_upper_D{n}'] = np.log(fe['band_upper']/close) / (fe['sigma'] * tf)
    fe[f'z_exact_lower_D{n}'] = np.log(fe['band_lower']/close) / (fe['sigma'] * tf)

    # 2. where the price sits inside the band, on a 0 to 1 scale
    fe[f'band_position_D{n}'] = (close - fe['band_lower']) / width

    # 3. how far off-centre the price is
    fe[f'band_asymmetry_D{n}'] = ((fe['band_upper']-close) - (close-fe['band_lower'])) / width

    # 4. recent volatility relative to the cycle sigma that built the band
    fe[f'vol_ratio_D{n}'] = fe[f'vol_D{n}'] / fe['sigma']

    # 5. is volatility rising or falling within the cycle
    fe[f'vol_momentum_D{n}'] = fe[f'vol_D{n}'] / fe['vol_D1']

    # 6. how much of the available room has already been used
    if f'cum_ret_D{n}' in fe.columns:
        fe[f'path_position_D{n}'] = fe[f'cum_ret_D{n}'] / fe['band_width_pct']

engineered = [c for c in fe.columns if c.startswith(('z_exact','band_position','band_asymmetry',
                                                     'vol_ratio','vol_momentum','path_position'))]
print(f'Engineered {len(engineered)} new features:')
for g, cols in pd.Series(engineered).groupby(lambda i: engineered[i].rsplit('_D',1)[0]):
    print(f'  {g:<20} {list(cols)}')

#### Observations:
- Twenty-something new features are created across six families, each derived purely from prices,
  the band levels and the volatility estimates already in use.
- The most conceptually important is **`z_exact`**, which is the Gaussian model's own sufficient
  statistic written out explicitly, including the square-root-of-time term that `norm_dist` omits.
  Making it available means a learned model can **reproduce the Gaussian exactly** and then
  deviate from it, rather than having to rediscover it from the component parts.

#### How well does each engineered feature separate the classes?

In [ ]:
rows = []
for col in engineered:
    target = 'lower_breach' if 'lower' in col else 'upper_breach'
    v = fe[col].replace([np.inf,-np.inf], np.nan)
    ok = v.notna()
    if ok.sum() < 50: continue
    yy, vv = fe.loc[ok, target].astype(int), v[ok]
    auc = roc_auc_score(yy, vv)
    auc_dir = max(auc, 1-auc)          # a sub-0.5 AUC just means the feature points the other way
    u, p = sps.mannwhitneyu(vv[yy==1], vv[yy==0], alternative='two-sided')
    rows.append({'feature': col, 'family': col.rsplit('_D',1)[0], 'target': target,
                 'AUC': round(auc_dir,4), 'p_value': p})

disc = pd.DataFrame(rows).sort_values('AUC', ascending=False)
disc.head(12)

In [ ]:
plt.figure(figsize=(9,4))
top = disc.head(12)
sns.barplot(data=top, y='feature', x='AUC', color='#2563eb')
plt.axvline(0.5, ls='--', color='#334155', label='no separation')
plt.axvline(0.60, ls=':', color='#dc2626', label='adoption threshold')
plt.xlim(0.45, 1.0); plt.legend(fontsize=8)
plt.title('Engineered features ranked by univariate separation')
plt.tight_layout(); plt.show()

# which family carries the signal?
fam = disc.groupby('family')['AUC'].agg(['mean','max','size']).sort_values('max', ascending=False)
fam.round(4)

#### Observations:
- The strongest engineered features reach an **AUC of about 0.95**, which is high — but this is
  measured at **D4**, one day before expiry, where the outcome is already largely determined.
- The **band-geometry family** (`band_position`, `band_asymmetry`) and the **standardised-distance
  family** (`z_exact`) are the strongest. The **volatility-regime family** is consistently the
  weakest.
- This mirrors the bivariate finding exactly: signal about a band breach comes from **where the
  price sits relative to the band**, not from the prevailing volatility level.
- A caution applies to interpreting these numbers. `band_position` and `band_asymmetry` are
  monotone transformations of one another and of `norm_dist`, so a high AUC does **not** mean they
  add information beyond what is already available. It confirms that distance is the dominant
  signal, which was already known.

#### Should these features actually be adopted? The events-per-variable constraint

In [ ]:
n_events = int(analysis['upper_breach'].sum())
n_train_events = int(round(n_events * 0.60))   # events available in a typical fitting block

epv = pd.DataFrame([
    {'feature_set': 'Minimal: exact-z only',        'k_features': 1},
    {'feature_set': 'Core day-only set',            'k_features': 6},
    {'feature_set': 'Cumulative set (carries D2-D3)','k_features': 16},
    {'feature_set': 'Core + all engineered',        'k_features': 6 + len(engineered)},
])
epv['events_per_variable'] = (n_train_events / epv['k_features']).round(1)
epv['verdict'] = np.where(epv['events_per_variable'] >= 10, 'Adequate', 'Below the recommended minimum')

plt.figure(figsize=(8,3.2))
ax = sns.barplot(data=epv, y='feature_set', x='events_per_variable', color='#2563eb')
plt.axvline(10, ls='--', color='#dc2626')
plt.text(10.5, 3.2, 'recommended minimum (10 EPV)', color='#dc2626', fontsize=8)
for c in ax.containers: ax.bar_label(c, fontsize=8)
plt.xlabel('events per variable'); plt.ylabel('')
plt.title('Why the feature set must be kept small')
plt.tight_layout(); plt.show()

print(f'Total breach events in the analysis sample : {n_events}')
print(f'Events available in a typical fitting block: {n_train_events}')
print()
epv

#### Observations:
- The analysis sample contains roughly **92 upper-breach events**, of which about **55** are
  available in a typical training block.
- The established guideline for logistic regression is a minimum of **10 to 20 events per
  variable**. Below that threshold, coefficient estimates become unstable and the model overfits.
- The core six-feature set sits at roughly **9 events per variable** — already marginal. The
  cumulative sixteen-feature set falls to about **3.4**, well below the guideline.
- **Adopting all the engineered features would make this substantially worse, not better.** Even
  though many of them separate the classes with a statistically significant margin, adding them
  costs degrees of freedom that this sample cannot afford.
- The correct conclusion is therefore the opposite of the intuitive one: the way to improve the
  learned model is to make it **smaller and better-specified**, not larger. The single most
  promising candidate is a minimal model built on `z_exact` alone, which nests the Gaussian in one
  parameter and enjoys roughly 55 events per variable.
- The one engineered feature retained for future work is **`z_exact`**, and specifically for a
  **pooled** model across D2, D3 and D4 — where the time term ceases to be constant and the
  feature becomes genuinely informative.

---
## Section 7: Minimum Sample Size and Statistical Power by Research Question

Each research question is supported by a formal sample-size calculation at the conventional
$\alpha = 0.05$ and power $= 0.80$. Three quantities are reported for each:

1. the **required N** at a stated effect size;
2. the **achieved N** actually available in this study; and
3. the **minimum detectable effect (MDE)** at the achieved N.

The third is the one that matters most for interpretation. Rather than asserting that a chosen
effect size is appropriate, it states the smallest effect the study is *capable* of detecting.
This also converts a null result from "nothing was found" into the far stronger claim that
"an effect of at least this size would have been detected, and was not."

In [ ]:
ALPHA, POWER = 0.05, 0.80
z_alpha = sps.norm.ppf(1 - ALPHA/2)
z_beta  = sps.norm.ppf(POWER)
print(f'z(1 - alpha/2) = {z_alpha:.4f}')
print(f'z(power)       = {z_beta:.4f}')

# the sample actually available, from the walk-forward design
N_CYCLES = len(analysis)
N_FOLDS, TEST_FRACTION = 4, 0.15
test_block = max(5, int(round(N_CYCLES * TEST_FRACTION)))
N_OOS = N_FOLDS * test_block

print()
print(f'Analysis cycles                     : {N_CYCLES}')
print(f'Walk-forward folds                  : {N_FOLDS}')
print(f'Test block per fold                 : {test_block} cycles')
print(f'Total out-of-sample cycles          : {N_OOS}')
print(f'Decision records (cycles x 3 days)  : {N_OOS*3}')
print(f'Upper-breach events out of sample   : {int(N_OOS*analysis["upper_breach"].mean())}')

#### RQ1: Can ML-core achieve a different F1 from the Gaussian?

In [ ]:
def cohen_h(p1, p2):
    """Effect size for the difference between two proportions."""
    return abs(2*np.arcsin(np.sqrt(p2)) - 2*np.arcsin(np.sqrt(p1)))

p1, p2 = 0.70, 0.80          # Gaussian baseline F1, and a practically meaningful target
h = cohen_h(p1, p2)
n_required = int(np.ceil(((z_alpha + z_beta) / h) ** 2))

# minimum detectable effect at the achieved sample size
h_mde = (z_alpha + z_beta) / np.sqrt(N_OOS)
p2_mde = np.sin(np.arcsin(np.sqrt(p1)) + h_mde/2) ** 2

print('Formula   h = |2*arcsin(sqrt(p2)) - 2*arcsin(sqrt(p1))|,   N = ((z_a + z_b)/h)^2')
print(f'Assumed   p1 = {p1} (Gaussian), p2 = {p2} (target)  ->  h = {h:.4f}')
print(f'REQUIRED  N = (({z_alpha:.3f} + {z_beta:.3f}) / {h:.4f})^2 = {n_required}')
print(f'ACHIEVED  N = {N_OOS}  ({N_OOS/n_required:.1f} times the requirement)')
print(f'MDE       h = {h_mde:.4f}, detectable from F1 {p1:.2f} to {p2_mde:.3f} '
      f'(a {100*(p2_mde-p1):.1f} percentage point improvement)')

#### Observations:
- The requirement is **146 paired decision records**; the study achieves **356**, comfortably
  **2.4 times** what is needed.
- The **minimum detectable effect is approximately 6.6 percentage points of F1**.
- Is that appropriate for this domain? Yes. A hold-or-exit decision on an option position is
  discrete — a model must change the decision often enough to alter the outcome. An improvement
  smaller than about five percentage points of F1 would change too few decisions to be
  operationally meaningful after transaction costs. The MDE therefore sits at the right order of
  magnitude, and the study is neither underpowered nor wastefully overpowered for RQ1.

#### RQ2: Are the Gaussian's errors systematic?

In [ ]:
BINS = 10
dof = BINS - 2

def chi2_required_n(w, dof, alpha=ALPHA, power=POWER):
    """Smallest N at which the non-central chi-square test reaches the target power."""
    crit = sps.chi2.ppf(1-alpha, dof)
    for n in range(30, 20001, 5):
        if 1 - sps.ncx2.cdf(crit, dof, n * w * w) >= power:
            return n
    return None

for w, label in [(0.10,'small'), (0.30,'medium'), (0.50,'large')]:
    print(f'  Cohen w = {w} ({label:<6}) -> required N = {chi2_required_n(w, dof)}')

crit = sps.chi2.ppf(1-ALPHA, dof)
achieved_power = 1 - sps.ncx2.cdf(crit, dof, N_OOS * 0.10**2)
w_mde = np.sqrt(min(l for l in np.arange(0.5, 60, 0.05)
                    if 1 - sps.ncx2.cdf(crit, dof, l) >= POWER) / N_OOS)
print()
print(f'ACHIEVED  N = {N_OOS}  ->  power against w = 0.10 is only {achieved_power:.3f}')
print(f'MDE       smallest detectable w at 80% power = {w_mde:.4f} (a medium effect)')

#### Observations:
- Detecting a **small** calibration error (Cohen $w = 0.10$) would require about **1,505**
  observations. The study has 356, giving only **21% power** against an effect that size.
- The study *is* adequately powered to detect a **medium** miscalibration ($w \approx 0.21$ or
  larger).
- This is an **honest limitation and is reported as such**. The practical implication is important
  for interpretation: if the calibration test returns a non-significant result, it should be read
  as "no *medium or large* miscalibration was detected", not as proof that the Gaussian is
  perfectly calibrated.
- A small miscalibration would in any case be of limited practical concern, since it would shift
  predicted probabilities by well under one percentage point per bin — not enough to change a
  hold-or-exit decision.

#### RQ3: Does predictability differ by decision day and direction?

In [ ]:
pa, pb = 0.65, 0.75
h3 = cohen_h(pa, pb)
n_req3 = int(np.ceil(2 * ((z_alpha + z_beta) / h3) ** 2))
h3_mde = (z_alpha + z_beta) * np.sqrt(2 / N_OOS)
pb_mde = np.sin(np.arcsin(np.sqrt(pa)) + h3_mde/2) ** 2

print('Formula   N per group = 2 * ((z_a + z_b)/h)^2   (two independent proportions)')
print(f'Assumed   p_a = {pa} (D2), p_b = {pb} (D4)  ->  h = {h3:.4f}')
print(f'REQUIRED  N = {n_req3} per group')
print(f'ACHIEVED  N = {N_OOS} per decision day')
print(f'MDE       h = {h3_mde:.4f}, i.e. {pa:.2f} versus {pb_mde:.3f} '
      f'({100*(pb_mde-pa):.1f} percentage points)')

#### Observations:
- The requirement is **328 observations per group**; the study has **356 per decision day**, so
  RQ3 is adequately but only **marginally** powered.
- The bivariate analysis in Section 4 already showed the day effect to be very large
  (separation growing from 0.77 to 1.45 sigma, all p-values below 10⁻¹⁷), so the **day comparison
  is not at risk**.
- The **direction comparison** (upper versus lower) is the binding case, since the two breach rates
  differ by only 0.5 percentage points. The study can detect a difference of about 10 percentage
  points, so a genuine but small directional asymmetry could go undetected. This is disclosed.

#### RQ4: Do the sigma level and look-back window change the result?

In [ ]:
SIGMA_GRID_PLANNED  = [0.5, 1.0]
WINDOW_GRID_PLANNED = [4, 8, 12, 16]
n_cells_planned = len(SIGMA_GRID_PLANNED) * len(WINDOW_GRID_PLANNED) * 6
n_cells_current = 1 * 2 * 6      # the interim run used one sigma and two windows

rho = 0.50
z_r = 0.5 * np.log((1+rho)/(1-rho))            # Fisher z transformation
n_req_rho = int(np.ceil(((z_alpha + z_beta)/z_r) ** 2 + 3))

print('Test      Spearman rank correlation between the band setting and model performance')
print(f'Formula   N = ((z_a + z_b) / z_r)^2 + 3,  where z_r = 0.5*ln((1+rho)/(1-rho))')
print(f'Assumed   rho = {rho} (a moderate association)  ->  z_r = {z_r:.4f}')
print(f'REQUIRED  N = {n_req_rho} configuration cells')
print()
print(f'ACHIEVED (interim run)  : {n_cells_current} cells  -> NOT sufficient')
print(f'PLANNED  (full grid)    : {n_cells_planned} cells  -> sufficient')

#### Observations:
- RQ4 is the **only research question whose sample is limited by the experimental design rather
  than by the data**. Each configuration cell is one observation for the correlation test, and
  cells are produced by running the grid, not by collecting more market history.
- A Spearman test at $\rho = 0.5$ requires about **30 cells**. The interim run covered a single
  sigma and two windows, giving only **12 cells** — insufficient for a formal test.
- The full planned grid of two sigma levels by four windows by six task cells gives **48**, which
  is comfortably sufficient.
- **This is therefore a resolvable limitation rather than a fundamental one**, and it is the
  single highest-value item of remaining work. It costs computation time, not additional data.
  For the interim report, RQ4 is presented **descriptively** through the configuration heatmap,
  with the formal correlation test deferred to the final report.

#### Sample size summary

In [ ]:
summary = pd.DataFrame([
    {'RQ':'RQ1','Test':'Two proportions (Cohen h)','Effect assumed':'p1=0.70, p2=0.80 (h=0.232)',
     'Required N':146,'Achieved N':N_OOS,'MDE at achieved N':'6.6 pp of F1','Adequate?':'Yes (2.4x)'},
    {'RQ':'RQ2','Test':'Hosmer-Lemeshow (10 bins)','Effect assumed':'Cohen w = 0.10 (small)',
     'Required N':1505,'Achieved N':N_OOS,'MDE at achieved N':'w = 0.21 (medium)','Adequate?':'Medium effects only'},
    {'RQ':'RQ3','Test':'Two independent proportions','Effect assumed':'0.65 vs 0.75 (h=0.219)',
     'Required N':328,'Achieved N':N_OOS,'MDE at achieved N':'9.6 pp','Adequate?':'Yes (marginal)'},
    {'RQ':'RQ4','Test':'Spearman across configurations','Effect assumed':'rho = 0.50',
     'Required N':30,'Achieved N':12,'MDE at achieved N':'grid-limited','Adequate?':'No - expand the grid'},
])
summary

---
## Section 8: Conclusions and Recommendations

### Conclusions

**On data quality**

1. The dataset is of high quality. Across 3,846 trading days spanning 15.6 years there are
   **no duplicates, no invalid prices, no gaps in the trading calendar**, and only nine missing
   values — all structural warm-up entries at the very start of the series.
2. **No imputation was performed anywhere.** Missing warm-up rows are excluded rather than filled,
   because forward-filling a price would fabricate a zero-return day and deflate the rolling
   volatility that defines the band widths.
3. Of 814 constructed cycles, **605 are standard** and form the basis of the study, reducing to
   **593** after the rolling-statistic warm-up is discarded.

**On the distribution of returns**

4. Daily returns are decisively **non-Normal**: skewness −0.93, excess kurtosis 14.14, and
   Jarque-Bera p ≈ 0. The worst day in the sample is a 13.4-sigma event under a Normal assumption.
5. However, **four-day cycle returns are much closer to Normal** than daily returns, consistent
   with the Central Limit Theorem. Since the study operates at the cycle horizon, the Gaussian
   assumption is far less violated than the daily statistics alone would suggest. This materially
   reframes RQ2.
6. **Direction is unpredictable but magnitude is persistent.** Return autocorrelation is
   insignificant at every lag, while absolute-return autocorrelation is significant out to lag 20.

**On the structure of the prediction problem**

7. The observed breach rate of **15.5% upper and 16.0% lower** matches the theoretical
   $1 - \Phi(1) = 15.87\%$ almost exactly, validating the band construction.
8. **Standardised distance to the band is by far the dominant predictor**, and its discriminative
   power rises sharply toward expiry — AUC of roughly 0.83 at D2 rising to 0.95 at D4. This
   directly answers RQ3.
9. **Volatility carries almost no residual information.** Breach rate is statistically independent
   of entry-day volatility (chi-square p = 0.48) despite volatility varying more than threefold
   across quintiles. The band is already volatility-scaled, so that signal has been consumed.
10. The feature space is **highly redundant**: two principal components explain about 90% of the
    variance in six features, and several VIF values exceed the conventional threshold.
11. Breach labels show **no serial dependence** across cycles, validating cycle-level resampling
    in the significance tests.

**On statistical power**

12. RQ1 and RQ3 are **adequately powered**. RQ2 can detect medium but not small miscalibration.
    RQ4 is **limited by the size of the configuration grid**, not by the data.
13. The events-per-variable analysis shows the binding constraint clearly. With roughly 55 breach
    events in a training block, the cumulative sixteen-feature set operates at about
    **3.4 events per variable** against a recommended minimum of ten.

### Recommendations

**For the modelling stage**

1. **Report the day-only feature set as the headline result.** It is the only configuration in
   which the identical-inputs claim strictly holds. The cumulative variant should be reported
   separately and clearly labelled.
2. **Prefer smaller, better-specified models over larger ones.** The events-per-variable
   arithmetic shows that this sample cannot support sixteen parameters. A minimal model built on
   the exact Gaussian z-statistic nests the analytical baseline in a single parameter and would
   operate at roughly 55 events per variable.
3. **Impose monotone constraints** on the tree ensembles. Breach probability must decrease as
   distance to the band increases, and the octile analysis confirms this relationship is monotone
   in the data. Enforcing it is a free reduction in variance that costs no observations.
4. **Use L2-regularised rather than unpenalised logistic regression**, given the severe
   multicollinearity documented in Section 5.
5. **Report each decision day separately.** Predictability differs so substantially between D2 and
   D4 that pooling the results would obscure both.

**For the remaining work**

6. **Expand the configuration grid to the full two sigma levels by four windows.** This is the
   single highest-value outstanding item: it converts RQ4 from a descriptive comparison into a
   formally testable one, and it costs computation rather than data.
7. **Evaluate a pooled D2/D3/D4 model** with `days_left` as a genuine feature. This would triple
   the row count, make the currently-constant time features informative, and allow the 208
   excluded holiday cycles to be recovered.
8. **Report calibration alongside classification.** Because the classes overlap substantially in
   every projection examined, the more interesting question may not be which model classifies
   better but which produces **trustworthy probabilities**. The Brier score decomposition into
   reliability and resolution addresses this directly.

**A note on expectations**

The exploratory analysis points consistently toward a limitation of **information** rather than of
**functional form**. Two principal components capture 90% of the feature variance; mutual
information is low for every predictor; volatility adds nothing once the band is volatility-scaled;
and the cycle-level return is much closer to Normal than the daily return. Taken together, these
suggest the Gaussian baseline may prove difficult to beat on these inputs.

That would be a **legitimate and publishable finding**, not a failure. Establishing that a widely
used analytical model is close to optimal on its own inputs — and demonstrating rigorously *why* —
is a genuine methodological contribution, and it points clearly to what a follow-on study would
need: richer information, not a more flexible function.